# Κεφάλαιο 8: Φτιάχνοντας τον tokenizer

> «Η γλώσσα είναι το ένδυμα της σκέψης.» — **Σάμιουελ Τζόνσον**, συγγραφέας

---

## Τι είναι η μετατροπή σε token;

**Μετατροπή σε token** σημαίνει ότι χωρίζουμε το κείμενο σε token (μονάδες κειμένου) και τα αντιστοιχίζουμε σε αριθμούς, ώστε να μπορούν να τα επεξεργαστούν τα νευρωνικά δίκτυα. Φαντάσου ότι μεταφράζεις τα αγγλικά σε έναν μυστικό κώδικα, όπου κάθε λέξη, τμήμα λέξης ή χαρακτήρας παίρνει έναν μοναδικό αριθμό.

```
"Hello world" → [15496, 995]  (με τον tokenizer του GPT-2)
```

Ο tokenizer λειτουργεί και αντίστροφα: του δίνεις αριθμούς και σου επιστρέφει κείμενο. Το αν παίρνεις πίσω το αρχικό κείμενο χωρίς αλλαγές εξαρτάται από τον tokenizer.

---

## Τι θα μάθεις

- Πώς το κείμενο γίνεται αριθμοί με τρεις διαφορετικές στρατηγικές μετατροπής σε token
- Γιατί η μετατροπή σε token ανά χαρακτήρα είναι απλή αλλά όχι αποδοτική
- Πώς η μετατροπή σε token ανά λέξη χειρίζεται τις άγνωστες λέξεις και γιατί έχει σημασία το μέγεθος του λεξιλογίου
- Ποιο έξυπνο τέχνασμα χρησιμοποιεί ο αλγόριθμος BPE για να φτιάξει token από τμήματα λέξεων και πώς αυτό βοηθά τα σύγχρονα μεγάλα γλωσσικά μοντέλα
- Πώς να χρησιμοποιείς έτοιμους tokenizer με εργαλεία όπως το tiktoken και το transformers της Hugging Face
- Ποιες ιδιαιτερότητες και παγίδες επηρεάζουν τις προτροπές σου και το κόστος χρήσης του API

---

## Προετοιμασία

Πρώτα, εγκατέστησε τα απαραίτητα πακέτα:

Άνοιξε το notebook [απευθείας στο Colab](https://colab.research.google.com/github/FirstLLM/code/blob/main/notebooks/el/ch08.ipynb) ή κατέβασε το [αρχείο για το Jupyter](https://github.com/FirstLLM/code/blob/main/notebooks/el/ch08.ipynb). Κρατάμε τα αγγλικά κείμενα των παραδειγμάτων ώστε οι είσοδοι, τα ID και οι ετικέτες τους να συμφωνούν.


In [ ]:
# Εγκατάσταση των απαραίτητων πακέτων
!pip install -q tiktoken==0.14.0 transformers==5.23.0 torch==2.15.0

## 1. Μετατροπή σε token ανά χαρακτήρα

Ας φτιάξουμε τον πιο απλό tokenizer: κάθε χαρακτήρας είναι ένα token.

**Υπενθύμιση για τις κλάσεις στην Python:**
- Μια **κλάση** είναι ένα πρότυπο που συνδυάζει δεδομένα και συναρτήσεις
- Η `__init__(self)` εκτελείται όταν δημιουργείς ένα αντικείμενο (αρχικοποιεί τα δεδομένα του)
- Το `self` αναφέρεται σε «αυτό το συγκεκριμένο αντικείμενο» (όπως όταν λες «αυτό το αυτοκίνητο» αντί για «τα αυτοκίνητα γενικά»)
- Το `@property` κάνει μια μέθοδο να συμπεριφέρεται σαν ιδιότητα (δεν χρειάζεσαι παρενθέσεις)

In [ ]:
class CharTokenizer:
    def __init__(self):
        # Δύο λεξικά για αναζήτηση και προς τις δύο κατευθύνσεις
        self.char_to_id = {}
        self.id_to_char = {}

    def fit(self, text):
        """Φτιάχνει το λεξιλόγιο από το κείμενο.
        
        Γιατί sorted(); Για να είναι σταθερή η αντιστοίχιση: το ίδιο κείμενο
        δίνει πάντα τα ίδια ID. Χωρίς ταξινόμηση, η σειρά σε ένα set της Python δεν είναι εγγυημένη.
        """
        # Συγκέντρωσε τους μοναδικούς χαρακτήρες: το σύνολο κρατά τον καθένα μόνο μία φορά
        chars = sorted(set(text))
        
        # Δώσε σε κάθε χαρακτήρα ένα ID (ξεκινώντας από το 0)
        for i, c in enumerate(chars):
            self.char_to_id[c] = i
            self.id_to_char[i] = c

    def encode(self, text):
        """Μετατρέπει το κείμενο σε λίστα με τα ID των token.
        
        Επιστρέφει μια λίστα ακεραίων, έναν για κάθε χαρακτήρα.
        """
        return [self.char_to_id[c] for c in text]

    def decode(self, ids):
        """Μετατρέπει τη λίστα με τα ID των token ξανά σε κείμενο.
        
        Χρησιμοποιεί τη str.join() για να ενώσει τους χαρακτήρες χωρίς να προσθέσει
        κενά ανάμεσά τους (ενώ όταν κάθε token είναι λέξη, χρειάζονται διαχωριστικά).
        """
        return "".join(self.id_to_char[i] for i in ids)

    @property
    def vocab_size(self):
        """Δίνει το πλήθος των μοναδικών χαρακτήρων που γνωρίζει ο tokenizer."""
        return len(self.char_to_id)

### Δοκίμασέ το: Πλήρες παράδειγμα

In [ ]:
# Δημιούργησε και εκπαίδευσε τον tokenizer
tokenizer = CharTokenizer()
tokenizer.fit("Hello, world!")

print(f"Μέγεθος λεξιλογίου: {tokenizer.vocab_size}")
print(f"Λεξιλόγιο: {tokenizer.char_to_id}")

# Κωδικοποίησε το κείμενο
text = "Hello"
ids = tokenizer.encode(text)
print(f"\n'{text}' → {ids}")

# Αποκωδικοποίησε τα ID: θα πρέπει να πάρεις πίσω το αρχικό κείμενο χωρίς απώλειες!
decoded = tokenizer.decode(ids)
print(f"{ids} → '{decoded}'")
print(f"Πήρες πίσω ακριβώς το αρχικό κείμενο; {text == decoded}")

### Ο συμβιβασμός ανάμεσα στο μέγεθος του λεξιλογίου και το μήκος της ακολουθίας

In [ ]:
sentence = "Tokenization is the first step in any language model."

# Σε επίπεδο χαρακτήρα
char_tok = CharTokenizer()
char_tok.fit(sentence)
char_ids = char_tok.encode(sentence)

print(f"Αρχικό κείμενο: {len(sentence)} χαρακτήρες")
print(f"Μέγεθος λεξιλογίου: {char_tok.vocab_size}")
print(f"Μήκος ακολουθίας: {len(char_ids)}")
print(f"Τα πρώτα 20 token: {char_ids[:20]}")

## 2. Χωρισμός σε token σε επίπεδο λέξης

Τώρα ας φτιάξουμε έναν tokenizer σε επίπεδο λέξης που χειρίζεται τις άγνωστες λέξεις με ειδικά token.

**Ειδικά token:** Έχουν δεσμευμένα αναγνωριστικά (ID) με συγκεκριμένη σημασία:
- `<PAD>` (ID 0): Συμπλήρωση — συμπληρώνει τις ακολουθίες ώστε να έχουν ίσο μήκος σε μια παρτίδα
- `<UNK>` (ID 1): Άγνωστη λέξη — αντιπροσωπεύει λέξεις που δεν υπάρχουν στο λεξιλόγιό μας
- `<BOS>` (ID 2): Αρχή ακολουθίας — δείχνει πού αρχίζει το κείμενο
- `<EOS>` (ID 3): Τέλος ακολουθίας — δείχνει πού τελειώνει το κείμενο

Γιατί τα χρειαζόμαστε; Χωρίς το `<UNK>`, ο tokenizer μας θα έδινε σφάλμα σε κάθε νέα λέξη. Χωρίς το `<PAD>`, δεν θα μπορούσαμε να βάλουμε προτάσεις διαφορετικού μήκους στην ίδια παρτίδα σταθερού μήκους.

In [ ]:
from collections import Counter

class WordTokenizer:
    def __init__(self, max_vocab_size=10000):
        """
        max_vocab_size: Μέγιστο μέγεθος λεξιλογίου (μαζί με τα ειδικά token)
        
        Γιατί 10.000; Είναι ένας συμβιβασμός:
        - Πολύ μικρό λεξιλόγιο (1.000): Πάρα πολλές άγνωστες λέξεις
        - Πολύ μεγάλο λεξιλόγιο (100.000): Τεράστιος πίνακας διανυσματικών αναπαραστάσεων (embedding), αργή εκπαίδευση
        - 10.000-50.000: Καλό εύρος για τη μάθηση
        """
        self.max_vocab_size = max_vocab_size
        self.word_to_id = {}
        self.id_to_word = {}

    def fit(self, text):
        """Φτιάξε το λεξιλόγιο από τις συχνότερες λέξεις."""
        # Ξεκίνα απλά: μετέτρεψε το κείμενο σε πεζά και χώρισέ το με βάση τους λευκούς χαρακτήρες
        words = text.lower().split()
        
        # Μέτρησε τις εμφανίσεις κάθε λέξης. Γιατί; Οι συχνές λέξεις παίρνουν δικό τους ID,
        # ενώ οι σπάνιες γίνονται <UNK>. Έτσι ελαχιστοποιείς τις εμφανίσεις του <UNK> στο κείμενο εκπαίδευσης.
        counts = Counter(words)
        
        # Δέσμευσε τα ID 0-3 για τα ειδικά token
        self.word_to_id = {
            "<PAD>": 0,
            "<UNK>": 1,
            "<BOS>": 2,
            "<EOS>": 3
        }
        self.id_to_word = {v: k for k, v in self.word_to_id.items()}
        
        # Πρόσθεσε τις συχνότερες λέξεις (χωρίς να ξεπεράσεις το όριο max_vocab_size)
        # Ξεκίνα τα ID από το 4, αφού τα 0-3 είναι δεσμευμένα για τα ειδικά token
        for i, (word, count) in enumerate(counts.most_common(self.max_vocab_size - 4), start=4):
            self.word_to_id[word] = i
            self.id_to_word[i] = word

    def encode(self, text, add_special_tokens=False):
        """Μετέτρεψε το κείμενο σε ID των token.
        
        add_special_tokens: Αν είναι True, πρόσθεσε <BOS> στην αρχή και <EOS> στο τέλος
        """
        words = text.lower().split()
        
        # Αναζήτησε κάθε λέξη· αν δεν τη βρεις, χρησιμοποίησε το <UNK> (ID=1)
        # Η .get(word, 1) επιστρέφει 1 αν η λέξη δεν υπάρχει στο λεξιλόγιό μας
        ids = [self.word_to_id.get(w, 1) for w in words]
        
        if add_special_tokens:
            ids = [2] + ids + [3]  # [<BOS>] + κείμενο + [<EOS>]
        
        return ids

    def decode(self, ids, skip_special_tokens=True):
        """Μετέτρεψε τα ID των token ξανά σε κείμενο.
        
        skip_special_tokens: Αν είναι True, μην εμφανίσεις τα <PAD>, <BOS> κτλ.
        Γιατί; Δεν θέλεις να βλέπεις αποτελέσματα όπως: "<BOS> Hello world <EOS>"
        """
        words = []
        for i in ids:
            word = self.id_to_word.get(i, "<UNK>")
            # Παράλειψε τα ειδικά token στην έξοδο αν έχεις επιλέξει να μην εμφανίζονται
            if skip_special_tokens and word in ["<PAD>", "<BOS>", "<EOS>"]:
                continue
            words.append(word)
        
        # Ένωσε τις λέξεις με κενά (σε αντίθεση με τον tokenizer χαρακτήρων που χρησιμοποιούσε "".join)
        return " ".join(words)

    @property
    def vocab_size(self):
        """Τρέχον μέγεθος λεξιλογίου (αριθμός διαφορετικών token)."""
        return len(self.word_to_id)

### Δοκίμασέ το: Πλήρες παράδειγμα με άγνωστες λέξεις

In [ ]:
# Φτιάξε έναν tokenizer με μικρό λεξιλόγιο ώστε κάποιες λέξεις να θεωρηθούν άγνωστες
tokenizer = WordTokenizer(max_vocab_size=10)

# Εκπαίδευσέ τον σε ένα μικρό δείγμα κειμένου
training_text = """
The cat sat on the mat.
The cat was on the mat.
The dog sat on the mat.
"""
tokenizer.fit(training_text)

print(f"Λεξιλόγιο: {tokenizer.word_to_id}")

# Κωδικοποίησε μια πρόταση με γνωστές λέξεις
text1 = "the cat sat"
ids1 = tokenizer.encode(text1)
print(f"\n'{text1}' → {ids1}")
print(f"Μετά την αποκωδικοποίηση: '{tokenizer.decode(ids1)}'")

# Κωδικοποίησε μια πρόταση με άγνωστη λέξη
text2 = "the elephant sat"  # Η λέξη "elephant" δεν υπάρχει στο λεξιλόγιο!
ids2 = tokenizer.encode(text2)
print(f"\n'{text2}' → {ids2}")
print(f"Μετά την αποκωδικοποίηση: '{tokenizer.decode(ids2)}'")

# Δοκίμασε και τα ειδικά token
ids3 = tokenizer.encode("the cat", add_special_tokens=True)
print(f"\nΜε ειδικά token: {ids3}")
print(f"Μετά την αποκωδικοποίηση (με εμφάνιση των ειδικών token): '{tokenizer.decode(ids3, skip_special_tokens=False)}'")
print(f"Μετά την αποκωδικοποίηση (με απόκρυψη των ειδικών token): '{tokenizer.decode(ids3, skip_special_tokens=True)}'")

### Άσκηση σύγκρισης: Δες τι κερδίζεις και τι χάνεις

In [ ]:
text = "The quick brown fox jumps over the lazy dog"

# Tokenizer χαρακτήρων
char_tok = CharTokenizer()
char_tok.fit(text)
char_ids = char_tok.encode(text)

# Tokenizer λέξεων
word_tok = WordTokenizer(max_vocab_size=20)
word_tok.fit(text)
word_ids = word_tok.encode(text)

print("TOKENIZER ΧΑΡΑΚΤΗΡΩΝ:")
print(f"  Μέγεθος λεξιλογίου: {char_tok.vocab_size}")
print(f"  Μήκος ακολουθίας: {len(char_ids)}")
print(f"  Token: {char_ids[:20]}...")

print("\nTOKENIZER ΛΕΞΕΩΝ:")
print(f"  Μέγεθος λεξιλογίου: {word_tok.vocab_size}")
print(f"  Μήκος ακολουθίας: {len(word_ids)}")
print(f"  Token: {word_ids}")

## Πώς λειτουργούν οι tokenizer στην πράξη: BPE (κωδικοποίηση ζευγών byte)

Τα σύγχρονα μεγάλα γλωσσικά μοντέλα χωρίζουν το κείμενο σε **υπολεκτικές μονάδες** — μια μέση λύση ανάμεσα στους χαρακτήρες και τις λέξεις:

**Το πρόβλημα:**
- Tokenizer χαρακτήρων: Παράγουν πάρα πολλά token (μονάδες κειμένου) ανά κείμενο, οπότε η επεξεργασία είναι αργή και δαπανηρή.
- Tokenizer λέξεων: Δεν μπορούν να χειριστούν λέξεις εκτός λεξιλογίου ("ChatGPT" → `<UNK>`).

**Η λύση: BPE (κωδικοποίηση ζευγών byte)**

Η μέθοδος BPE μαθαίνει αυτόματα υπολεκτικές μονάδες: ξεκινά από χαρακτήρες ή byte και σε κάθε βήμα συγχωνεύει το συχνότερο ζεύγος γειτονικών μονάδων στα δεδομένα εκπαίδευσης:

```
Βήμα 1: Ξεκίνα με χαρακτήρες: ["l", "o", "w", "e", "r"]
Βήμα 2: Το συχνότερο ζεύγος είναι ("l", "o") → συγχώνευσέ το σε "lo"
Βήμα 3: Το συχνότερο ζεύγος είναι ("lo", "w") → συγχώνευσέ το σε "low"
Βήμα 4: Συνέχισε μέχρι το λεξιλόγιο να φτάσει στο επιθυμητό μέγεθος...
```

**Το αποτέλεσμα:** Οι συχνές λέξεις μπορούν να γίνουν μεμονωμένα token, ενώ οι σπάνιες χωρίζονται σε γνωστά κομμάτια. Για παράδειγμα:
- "lower" → ["low", "er"] ✓ (συχνά κομμάτια, αποδοτική κωδικοποίηση)
- "lowest" → ["low", "est"] ✓ (αναλύεται και ο υπερθετικός!)
- "ChatGPT" → ["Chat", "G", "PT"] ✓ (δεν χρειάζεται `<UNK>`!)

**Να θυμάσαι:** Μόνο η μέθοδος BPE σε επίπεδο byte με πλήρες αλφάβητο byte εγγυάται ότι δεν θα χρειαστεί `<UNK>`: κάθε κείμενο μπορεί να αναπαρασταθεί ως ακολουθία γνωστών byte.

---

## 3. Tokenizer στην πράξη: tiktoken

Δοκίμασε τώρα τη βιβλιοθήκη tiktoken της OpenAI με το `cl100k_base`, το σχήμα κωδικοποίησης της εποχής των GPT-3.5/GPT-4 (2023). Νεότερα μοντέλα της OpenAI χρησιμοποιούν το `o200k_base`, που έχει μεγαλύτερο λεξιλόγιο· ο μηχανισμός είναι ο ίδιος.

## Εκπαίδευσε τον δικό σου tokenizer BPE

Δες τώρα πώς μαθαίνει η BPE! Θα εκπαιδεύσεις τον δικό σου tokenizer σε κείμενα του Σαίξπηρ με τη βιβλιοθήκη `tokenizers` της Hugging Face.

**Γιατί να το κάνεις;**
- Για να δεις πώς προκύπτουν αυτόματα μοτίβα ("ing", "the", "tion").
- Για να καταλάβεις στην πράξη πώς λειτουργούν οι tokenizer.
- Για να δεις τι συμβαίνει όταν το κείμενο διαφέρει από τα δεδομένα εκπαίδευσης (π.χ. όταν εφαρμόζεις σε σύγχρονο κείμενο έναν tokenizer που εκπαιδεύτηκε σε έργα του Σαίξπηρ).

Πρώτα, εγκατάστησε τη βιβλιοθήκη `tokenizers`:

In [ ]:
# Εγκατάστησε τη βιβλιοθήκη tokenizers (είναι ξεχωριστή από τη transformers!)
!pip install -q tokenizers

In [ ]:
from tokenizers import Tokenizer
from tokenizers.models import BPE
from tokenizers.trainers import BpeTrainer
from tokenizers.pre_tokenizers import Whitespace

print("Οι εισαγωγές ολοκληρώθηκαν με επιτυχία!")

### Κατέβασε το TinyShakespeare

Θα χρησιμοποιήσουμε το ίδιο σύνολο δεδομένων με το οποίο θα εκπαιδεύσεις το πλήρες γλωσσικό σου μοντέλο (LLM) στο Κεφάλαιο 12:

In [ ]:
import urllib.request

# Κατέβασε το TinyShakespeare (το ίδιο σύνολο δεδομένων που θα χρησιμοποιήσουμε στο Κεφάλαιο 12!)
url = "https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt"
urllib.request.urlretrieve(url, "shakespeare.txt")

# Διάβασε το κείμενο και ρίξε μια ματιά
with open("shakespeare.txt", "r") as f:
    text = f.read()

print(f"Μέγεθος συνόλου δεδομένων: {len(text):,} χαρακτήρες")
print(f"\nΟι πρώτοι 200 χαρακτήρες:")
print(text[:200])

### Εκπαίδευσε τον tokenizer BPE

Τώρα αρχίζει το συναρπαστικό κομμάτι: δες πώς ο αλγόριθμος BPE μαθαίνει μοτίβα από τα κείμενα του Σαίξπηρ!

In [ ]:
# Δημιούργησε έναν κενό tokenizer BPE
bpe_tokenizer = Tokenizer(BPE(unk_token="<UNK>"))

# Χώρισε το κείμενο με βάση τα κενά διαστήματα πριν από τις συγχωνεύσεις BPE
bpe_tokenizer.pre_tokenizer = Whitespace()

# Όρισε τις παραμέτρους εκπαίδευσης
trainer = BpeTrainer(
    vocab_size=1000,           # Μικρό λεξιλόγιο για εξάσκηση (στην παραγωγή χρησιμοποιούνται πάνω από 30.000 token)
    min_frequency=2,           # Ένα ζεύγος token πρέπει να εμφανίζεται τουλάχιστον δύο φορές για να συγχωνευτεί
    special_tokens=["<PAD>", "<UNK>", "<BOS>", "<EOS>"]
)

# Εκπαίδευσε τον tokenizer σε κείμενα του Σαίξπηρ (χρειάζονται μόνο λίγα δευτερόλεπτα!)
bpe_tokenizer.train(files=["shakespeare.txt"], trainer=trainer)

print(f"Μέγεθος λεξιλογίου: {bpe_tokenizer.get_vocab_size()}")

### Δοκίμασε τον tokenizer που εκπαίδευσες

In [ ]:
# Δοκίμασε τον tokenizer σε κείμενο στο ύφος του Σαίξπηρ
test_text = "To be or not to be, that is the question."
output = bpe_tokenizer.encode(test_text)

print(f"Κείμενο: '{test_text}'")
print(f"Μονάδες κειμένου (token): {output.tokens}")
print(f"Αναγνωριστικά των token: {output.ids}")
print(f"Αριθμός token: {len(output.ids)}")

# Έλεγξε αν η αποκωδικοποίηση επαναφέρει ακριβώς το αρχικό κείμενο
decoded = bpe_tokenizer.decode(output.ids)
print(f"\nΑποκωδικοποιημένο κείμενο: '{decoded}'")
print(f"Πιστή αναπαραγωγή του αρχικού κειμένου: {test_text == decoded}")

# Έλεγξε ότι τα αναγνωριστικά που δίνει ο tokenizer παραμένουν ίδια μετά την αποθήκευση και την επαναφόρτωση,
# παρόλο που ο προδιαχωρισμός με βάση τα κενά δεν διατηρεί τα αρχικά διαστήματα του κειμένου.
known_text = "To be or not to be"
known_ids = bpe_tokenizer.encode(known_text).ids
bpe_tokenizer.save("shakespeare_bpe.json")
reloaded = Tokenizer.from_file("shakespeare_bpe.json")
assert reloaded.encode(known_text).ids == known_ids
assert reloaded.get_vocab_size() == bpe_tokenizer.get_vocab_size()
print(f"Ο έλεγχος επαναφόρτωσης πέτυχε: {known_text!r} -> {known_ids}")

### Αναντιστοιχία πεδίου: σύγχρονο κείμενο σε tokenizer εκπαιδευμένο σε έργα του Σαίξπηρ

Τι συμβαίνει αν δοκιμάσεις ένα σύγχρονο τεχνικό κείμενο που ο Σαίξπηρ δεν έγραψε ποτέ;

In [ ]:
# Δοκίμασε ένα σύγχρονο κείμενο που ο Σαίξπηρ δεν έγραψε ποτέ
modern_text = "ChatGPT generates amazing neural network responses"
output2 = bpe_tokenizer.encode(modern_text)

print(f"Κείμενο: '{modern_text}'")
print(f"Token: {output2.tokens}")
print(f"Αριθμός token: {len(output2.ids)}")

# Πρόσεξε: το "ing" είναι ένα token (το χρησιμοποιούσε ο Σαίξπηρ!), αλλά το "ChatGPT" χωρίζεται σε κομμάτια
print("\nΤι να κρατήσεις:")
print("- Το 'ing' είναι ένα μόνο token (ο Σαίξπηρ χρησιμοποιούσε συχνά λέξεις με κατάληξη '-ing')")
print("- Το 'ChatGPT' χωρίζεται σε κομμάτια (ο Σαίξπηρ δεν έγραψε ποτέ για την τεχνητή νοημοσύνη!)")
print("- Έτσι φαίνεται στην πράξη η αναντιστοιχία πεδίου")

---

## Οι tokenizer που χρησιμοποιούνται στην παραγωγή: tiktoken

Τώρα είδες πώς εκπαιδεύεται ένας tokenizer BPE από το μηδέν. Ξέρεις πια πώς λειτουργούν οι tokenizer που χρησιμοποιούνται στην παραγωγή. Όταν το tiktoken χωρίζει το "ChatGPT" σε κομμάτια, ξέρεις γιατί!

Ας δοκιμάσουμε τώρα τα εργαλεία παραγωγής που θα χρησιμοποιείς στα έργα σου.

In [ ]:
import tiktoken

# Φόρτωσε την κωδικοποίηση του tokenizer της εποχής των GPT-3.5/GPT-4 (2023)
enc = tiktoken.get_encoding("cl100k_base")  # Εποχή των GPT-3.5/GPT-4· νεότερα μοντέλα της OpenAI χρησιμοποιούν την o200k_base

# Κωδικοποίησε το κείμενο
text = "Hello, world! How are you?"
tokens = enc.encode(text)

print(f"Κείμενο: '{text}'")
print(f"Token: {tokens}")
print(f"Αριθμός token: {len(tokens)}")

# Αποκωδικοποίησε τα token για να πάρεις ξανά το κείμενο
decoded = enc.decode(tokens)
print(f"Αποκωδικοποιημένο κείμενο: '{decoded}'")
print(f"Ακριβής ανάκτηση του αρχικού κειμένου: {text == decoded}")

### Δες τις συμβολοσειρές που αντιστοιχούν στα token

In [ ]:
# Αποκωδικοποίησε κάθε token ξεχωριστά για να δεις τι παριστάνει
token_strings = [enc.decode([t]) for t in tokens]

print(f"Ανάλυση ανά token:")
for token_id, token_str in zip(tokens, token_strings):
    print(f"  {token_id:5d} → '{token_str}'")

### Μέτρηση των token για τον υπολογισμό του κόστους χρήσης του API

In [ ]:
prompts = [
    "Write a haiku about programming.",
    "Explain quantum computing in simple terms for a 10-year-old child.",
    "Generate a 500-word essay on climate change."
]

enc = tiktoken.get_encoding("o200k_base")
# Ενδεικτική χρέωση εισόδου από το κεφάλαιο· έλεγξε την τρέχουσα τιμή για το συγκεκριμένο endpoint.
price_per_1m_input = 0.10

for prompt in prompts:
    tokens = enc.encode(prompt)
    cost = len(tokens) / 1_000_000 * price_per_1m_input
    
    print(f"Προτροπή: '{prompt}'")
    print(f"  Πλήθος token: {len(tokens)}")
    print(f"  Κόστος (είσοδος): ~${cost:.7f}\n")

## 4. Tokenizer για χρήση στην παραγωγή: Hugging Face Transformers

Η **Hugging Face** είναι η εταιρεία πίσω από τη βιβλιοθήκη Transformers, η οποία παρέχει προεκπαιδευμένα μοντέλα και tokenizer. Το `AutoTokenizer` φορτώνει αυτόματα το κατάλληλο tokenizer για κάθε υποστηριζόμενο μοντέλο.

Το tokenizer της Hugging Face επιστρέφει ένα λεξικό με τα εξής πεδία:
- `input_ids`: Τα αναγνωριστικά των token (αυτά μας ενδιαφέρουν περισσότερο)
- `attention_mask`: 1 για τα πραγματικά token, 0 για τις θέσεις συμπλήρωσης (δείχνει στο μοντέλο τι να αγνοήσει)

In [ ]:
from transformers import AutoTokenizer

# Φόρτωσε το tokenizer του GPT-2 (ανοιχτού κώδικα)
tokenizer = AutoTokenizer.from_pretrained("gpt2")

text = "Hello, world! How are you?"

# Κωδικοποίησε το κείμενο — θα πάρεις ένα λεξικό με τα αναγνωριστικά των token και τη μάσκα προσοχής
encoded = tokenizer(text, return_tensors="pt")  # "pt" = τανυστές PyTorch

print(f"Κείμενο: '{text}'")
print(f"Αναγνωριστικά των token: {encoded['input_ids']}")
print(f"Μάσκα προσοχής: {encoded['attention_mask']}")

# Αποκωδικοποίησε τα token
decoded = tokenizer.decode(encoded['input_ids'][0])
print(f"Αποκωδικοποιημένο κείμενο: '{decoded}'")

### Ανάλυση ανά token

In [ ]:
tokens = encoded['input_ids'][0].tolist()
token_strings = [tokenizer.decode([t]) for t in tokens]

print(f"Ανάλυση ανά token:")
for tid, tstr in zip(tokens, token_strings):
    print(f"  {tid:5d} → '{tstr}'")

## 5. Ιδιαιτερότητες της κατάτμησης σε token

### Ιδιαιτερότητα #1: Ένα κενό στην αρχή αλλάζει τα πάντα

In [ ]:
import tiktoken

enc = tiktoken.get_encoding("cl100k_base")

# Σύγκρινε τις λέξεις με και χωρίς κενό στην αρχή
texts = ["Hello", " Hello", "world", " world"]

for text in texts:
    tokens = enc.encode(text)
    print(f"'{text}' → {tokens} ({len(tokens)} token)")

### Ιδιαιτερότητα #2: Οι αριθμοί χωρίζονται σε ομάδες ψηφίων

In [ ]:
numbers = ["10", "100", "1000", "10000", "42", "2024"]

enc = tiktoken.get_encoding("cl100k_base")

for num in numbers:
    tokens = enc.encode(num)
    token_strs = [enc.decode([t]) for t in tokens]
    print(f"'{num}' → {tokens} = {token_strs}")

### Ιδιαιτερότητα #3: Emoji και ειδικοί χαρακτήρες

In [ ]:
emojis = ["😀", "🚀", "👍", "Hello 😀 world", "🔥🔥🔥"]

enc = tiktoken.get_encoding("cl100k_base")

for text in emojis:
    tokens = enc.encode(text)
    token_strs = [enc.decode([t]) for t in tokens]
    print(f"'{text}' → {len(tokens)} token: {token_strs}")

## 6. Πρακτική άσκηση: Χώρισε το σύνολο δεδομένων σου σε token

Ώρα να συνδυάσεις όσα έμαθες. Πάρε το `train.jsonl`, το αρχείο που αποθήκευσες στο notebook του Κεφαλαίου 7, και χώρισε το κείμενο κάθε εγγραφής σε token (μονάδες κειμένου) με τον tokenizer του GPT-2. Γιατί να χρησιμοποιήσεις τον tokenizer του GPT-2 και όχι το tiktoken; Επειδή στα Κεφάλαια 9-12 θα φτιάξεις ένα μοντέλο τύπου GPT-2 με το ίδιο λεξιλόγιο των 50.257 token. Έτσι, τα ID που θα αποθηκεύσεις εδώ είναι ακριβώς αυτά που μπορεί να χρησιμοποιήσει το μοντέλο.

**Πώς να φέρεις το αρχείο σε ΑΥΤΟ το περιβάλλον εκτέλεσης.** Κάθε notebook του Colab εκτελείται στο δικό του, καινούργιο περιβάλλον, οπότε το `train.jsonl` του Κεφαλαίου 7 ΔΕΝ μεταφέρεται αυτόματα. Διάλεξε έναν από τους δύο τρόπους:

1. **Λήψη / μεταφόρτωση:** στο notebook του Κεφαλαίου 7 εκτέλεσε `from google.colab import files; files.download('train.jsonl')` και μετά, σε αυτό το notebook, εκτέλεσε `from google.colab import files; files.upload()` και επίλεξε το αρχείο.
2. **Εκτέλεσε ξανά το Κεφάλαιο 7 εδώ:** αντίγραψε τα κελιά του Κεφαλαίου 7 σε αυτό το notebook και εκτέλεσέ τα με τη σειρά μέχρι και την αποθήκευση του αρχείου, ώστε το `train.jsonl` να γραφτεί στον τρέχοντα φάκελο εργασίας.

Στον δικό σου υπολογιστή, αν τα notebook χρησιμοποιούν τον ίδιο φάκελο εργασίας, έχουν πρόσβαση στο ίδιο αρχείο· δεν χρειάζεται μεταφορά. Το `chapter8_output.jsonl` κρατά τα `text`, `source`, `split` και `length` κάθε εγγραφής και προσθέτει τα `token_ids` και `token_count`. Για τις τρεις εγγραφές του παραδείγματος θα βρεις 7 token σε καθεμία. Πρόκειται για παράδειγμα μορφής αρχείου, όχι για το σώμα εκπαίδευσης του Κεφαλαίου 12.


In [ ]:
import json
from transformers import GPT2TokenizerFast

# Φόρτωσε το σύνολο δεδομένων σου από το Κεφάλαιο 7. Στο Colab τα περιβάλλοντα
# εκτέλεσης είναι ξεχωριστά: κατέβασε το train.jsonl από το notebook του Κεφαλαίου 7
# και ανέβασέ το εδώ (ή εκτέλεσε ξανά εδώ τα βήματα επεξεργασίας του Κεφαλαίου 7).
# Δες τις οδηγίες στο παραπάνω κελί κειμένου.
import os
if not os.path.exists("train.jsonl"):
    raise FileNotFoundError(
        "Το train.jsonl δεν βρέθηκε σε αυτό το περιβάλλον εκτέλεσης. Στο Colab, κατέβασέ το από "
        "το notebook του Κεφαλαίου 7 (files.download('train.jsonl')) και ανέβασέ "
        "το εδώ (files.upload()), ή εκτέλεσε πρώτα τα κελιά επεξεργασίας του Κεφαλαίου 7 "
        "σε αυτό το περιβάλλον.")
with open("train.jsonl", "r", encoding="utf-8") as f:
    dataset = [json.loads(line) for line in f]

# Ο tokenizer του GPT-2: το ίδιο λεξιλόγιο που θα χρησιμοποιήσεις στα Κεφάλαια 9-12
tokenizer = GPT2TokenizerFast.from_pretrained("gpt2")

for record in dataset:
    tokens = tokenizer.encode(record["text"])
    record["token_ids"] = tokens
    record["token_count"] = len(tokens)

# Αποθήκευσε το σύνολο δεδομένων με τα token· οι ετικέτες διαχωρισμού παραμένουν άθικτες
with open("chapter8_output.jsonl", "w", encoding="utf-8") as f:
    for record in dataset:
        f.write(json.dumps(record, ensure_ascii=False) + "\n")

print(f"Πλήθος παραδειγμάτων που χωρίστηκαν σε token: {len(dataset)}")

token_counts = [r["token_count"] for r in dataset]
avg_tokens = sum(token_counts) / len(token_counts)
max_tokens = max(token_counts)
min_tokens = min(token_counts)

print(f"\nΣτατιστικά:")
print(f"  Μέσο πλήθος token ανά παράδειγμα: {avg_tokens:.1f}")
print(f"  Μέγιστο πλήθος token: {max_tokens}")
print(f"  Ελάχιστο πλήθος token: {min_tokens}")

# Σύγκρινε τον tokenizer του GPT-2 με μια κωδικοποίηση που χρησιμοποιούσαν μοντέλα GPT-3.5/GPT-4.
import tiktoken

enc = tiktoken.get_encoding("cl100k_base")
text = dataset[0]["text"]
print(f"ID του GPT-2: {tokenizer.encode(text)}")
print(f"ID του cl100k: {enc.encode(text)}")

# Αποθηκεύσαμε τα ID του GPT-2 επειδή είναι δείκτες στον πίνακα διανυσματικών αναπαραστάσεων του MiniGPT.
# Για εκτιμήσεις χρήσης του API, χρησιμοποίησε τον tokenizer του αντίστοιχου endpoint. Για την πραγματική
# χρήση token, διάβασε το πεδίο usage της απόκρισης. Η ιστορική κωδικοποίηση cl100k_base δεν καθορίζει τη χρέωση.


## Σύνοψη κεφαλαίου

**Τι έφτιαξες και τι χρησιμοποίησες:**

1. **Tokenizer χαρακτήρων:** Απλός αλλά όχι αποδοτικός (μικροσκοπικό λεξιλόγιο ~100 χαρακτήρων, μεγάλες ακολουθίες)
2. **Tokenizer λέξεων:** Σύντομες ακολουθίες, αλλά τεράστιο λεξιλόγιο και προβλήματα με τις άγνωστες λέξεις
3. **Tokenizer BPE:** Εκπαίδευσες τον δικό σου σε κείμενα του Σαίξπηρ και είδες μοτίβα να αναδύονται αυτόματα!
4. **Εργαλεία παραγωγής:** Χρησιμοποίησες το tiktoken και εργαλεία του Hugging Face για να χωρίσεις κείμενο σε token στην πράξη

**Τι έμαθες:**

- Το αν μπορείς να ανακτήσεις ακριβώς το αρχικό κείμενο από τα token εξαρτάται από τον tokenizer
- Η σχέση ανάμεσα στο μέγεθος του λεξιλογίου και το μήκος της ακολουθίας είναι θεμελιώδης: ένα μεγαλύτερο λεξιλόγιο επιτρέπει συνήθως μικρότερες ακολουθίες
- Τα ειδικά token εξυπηρετούν συγκεκριμένους σκοπούς (BOS/EOS/PAD/UNK)
- Το αλγόριθμος BPE μαθαίνει αυτόματα τμήματα λέξεων συγχωνεύοντας συχνά ζεύγη· μόνο η μέθοδος BPE σε επίπεδο byte με πλήρες αλφάβητο byte εγγυάται ότι δεν θα χρειαστεί token για άγνωστα σύμβολα
- Έχει σημασία σε τι είδους κείμενα εκπαιδεύτηκε ο tokenizer: ο tokenizer που εκπαίδευσες στον Σαίξπηρ δυσκολεύεται με τους σύγχρονους τεχνολογικούς όρους
- Ο χωρισμός σε token έχει ιδιαιτερότητες (κενά στην αρχή, διάσπαση αριθμών, emoji)

**Στη συνέχεια:** Στο Κεφάλαιο 9 θα μετατρέψεις αυτά τα ID των token σε διανυσματικές αναπαραστάσεις (embedding)!